In [ ]:
import os
import json
import torch
import random
import shutil
import numpy as np

from pathlib import Path

from src.players.build_players import build_players
from src.game.controller import GameController
from src.game.game_logger import write_hand_rows
from src.players.snapshot import algo_of_player, snapshot_player_memory, run_generalization_table
from src.eval import memory_eval

In [ ]:
# ========================= 设置参数 =========================
# exp1_memory 的目标:
# 在同一模型下，不同 memory 机制的性能比较/策略涌现差异
EXP_NAME = "exp1_memory"

# 算法: fact / expr / fxsync / fxasync / naive
# 模型: deepseekv4f / deepseekv4p
PLAYERS = [
    "fact-deepseekv4f",
    "expr-deepseekv4f",
    "fxsync-deepseekv4f",
    "fact-deepseekv4f",
    "expr-deepseekv4f",
    "fxsync-deepseekv4f",
]

# 训练参数
TRAIN_HANDS = 50  # 训练轮数
STARTING_STACK = 1000  # 初始筹码数
RESUME = False  # 跨场续载

# 评估参数
SNAPSHOT_EVERY = 10  # 多少轮保存一次 memory snapshot + 跑一次泛化评估
TEST_HANDS = 10  # 测试轮数
NUM_NAIVE_PLAYERS = 5  # 测试玩家数量
NAIVE_LLM = "deepseekv4f"  # 测试模型

# 其他参数
OUTPUT_DIR = f"./output/{EXP_NAME}/h{TRAIN_HANDS}_nonabstract"
SEED = 42

In [ ]:
# ========================= 设置种子 =========================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.mps.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
# ========================= 构建玩家 =========================
if not RESUME and os.path.exists(OUTPUT_DIR):
    raise FileExistsError(f"{OUTPUT_DIR} already exists")

players = build_players(
    player_names=PLAYERS,
    starting_stack=STARTING_STACK,
    output_dir=OUTPUT_DIR,
)
for p in players:
    print(p.player_id, type(p).__name__)

In [ ]:
# ========================= 构建牌桌 =========================
controller = GameController(players=players)

SNAPSHOTS_DIR = Path(OUTPUT_DIR) / "_snapshots"
SNAPSHOTS_DIR.mkdir(parents=True, exist_ok=True)
snapshot_records = []   # [{"hand": h, "player_id": pid, "algo": "fact", "snapshot_dir": "..."}]

In [ ]:
# ========================= 开始游戏（主桌）=========================
print("Self-Evolving...")
for hand_idx in range(1, TRAIN_HANDS + 1):
    if sum(1 for p in players if p.stack > 0) < 2:
        break

    controller.start_hand()
    while not controller.hand_finished:
        seat = controller.current_player_seat
        if seat is None:
            break
        current_player = controller.players_by_seat[seat]
        state = controller.get_state(viewer_id=current_player.player_id)
        action = current_player.decide(state)
        controller.apply_action(action)

    final_state = controller.get_state(viewer_id=None)
    for p in players:
        p.observe(final_state)
    write_hand_rows(OUTPUT_DIR, controller, final_state)

    # ===== RECORDING: methodology (人读) + memory snapshot (机读+留底) =====
    for p in players:
        algo = algo_of_player(p)
        if algo is None:
            continue

        if hasattr(p, "extract_methodology"):
            md_path = Path(OUTPUT_DIR) / p.player_id / f"methodology_h{hand_idx:03d}.md"
            md_path.parent.mkdir(parents=True, exist_ok=True)
            md_path.write_text(p.extract_methodology(), encoding="utf-8")

        if hand_idx % SNAPSHOT_EVERY == 0:
            snap_dir = SNAPSHOTS_DIR / f"h{hand_idx:03d}" / p.player_id
            snapshot_player_memory(p, snap_dir)
            snapshot_records.append({
                "hand":          hand_idx,
                "player_id":     p.player_id,
                "algo":          algo,
                "snapshot_dir":  str(snap_dir),
            })

In [ ]:
# ========================= 泛化评估 =========================
# 对每个 snapshot，复活成一个新 agent，与 N_NAIVE_OPPONENTS 个 NaiveLLM 同桌打 GEN_HANDS 手。
# 同一 algo 多个起点 + 多个 sub_hand 之后会在 plot 里取均值。
GEN_OUT_DIR = Path(OUTPUT_DIR) / "_generalization"
if GEN_OUT_DIR.exists():
    shutil.rmtree(GEN_OUT_DIR)
GEN_OUT_DIR.mkdir(parents=True, exist_ok=True)

generalization_results = []
for rec in snapshot_records:
    sub_dir = GEN_OUT_DIR / f"h{rec['hand']:03d}" / rec["player_id"]
    print(f"  generalizing {rec['player_id']} @ h{rec['hand']:03d} (algo={rec['algo']})")
    # 通过 player_id 的后缀拿到原模型名（"fact-deepseekv4f" → "deepseekv4f"）
    pname_tail = rec["player_id"].split("_", 2)[-1]
    model_name = pname_tail.split("-", 1)[1]

    rows = run_generalization_table(
        snapshot_dir=rec["snapshot_dir"],
        algo=rec["algo"],
        model_name=model_name,
        player_id_in_snap=rec["player_id"],
        new_output_dir=sub_dir,
        n_opponents=NUM_NAIVE_PLAYERS,
        hands=TEST_HANDS,
        starting_stack=STARTING_STACK,
        naive_model=NAIVE_LLM,
    )
    generalization_results.append({
        "checkpoint_hand": rec["hand"],
        "algo":            rec["algo"],
        "src_pid":         rec["player_id"],
        "rows":            rows,
    })

# 留底
with open(GEN_OUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(generalization_results, f, ensure_ascii=False, indent=2)

In [ ]:
# ========================= Plot 1: 主桌 avg stack by algo =========================
EVAL_SAVE_DIR = Path(OUTPUT_DIR) / "_eval"
EVAL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
memory_eval.plot_avg_stack_by_algo(
    OUTPUT_DIR,
    save_path=EVAL_SAVE_DIR / "avg_stack_by_algo.png",
)

In [ ]:
# ========================= Plot 2: 泛化桌 avg stack by algo =========================
memory_eval.plot_generalization_by_algo(
    generalization_results,
    save_path=EVAL_SAVE_DIR / "generalization_by_algo.png",
)

In [ ]:
# ========================= Plot 3: 诈唬率 by algo =========================
# 诈唬定义（代理）：本手有 raise 动作，且 (a) 最终没赢，或 (b) 虽赢但牌型为纯 High Card。
# 单位为 algo 维度——把同算法多个玩家聚合后做滑窗。
memory_eval.plot_bluff_rate_by_algo(
    OUTPUT_DIR, window=10,
    save_path=EVAL_SAVE_DIR / "bluff_rate_by_algo.png",
)

In [ ]:
# ========================= 一键 report =========================
memory_eval.report(
    OUTPUT_DIR,
    generalization_results=generalization_results,
    save_dir=EVAL_SAVE_DIR,
    window=10,
)